# 02. Indexing, Slicing & Boolean Masking: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **02. Indexing, Slicing & Boolean Masking**. Efficient data manipulation in NumPy relies on understanding the distinction between zero-copy memory views (created via basic slicing `start:stop:step` and the ellipsis `...`) and independent memory allocations (created via fancy integer indexing and boolean masking). This notebook covers multidimensional coordinate slicing, view validation with `.base`, vectorized boolean predicate masks, open mesh indexing with `np.ix_()`, coordinate extraction via `np.nonzero()`, and indexed assignment with `np.take()` and `np.put()`.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 D Slicing: `arr[start:stop:step]`
- [x] 🔹 Multi-Axis 2D Slicing
- [x] 🔹 View Checking with `.base`
- [x] 🔹 Vectorized Boolean Masking
- [x] 🔹 Fancy Indexing with Integer Arrays


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14251 clean aligned rows):
- amounts array: shape (14251,), dtype float64
- fraud_flags array: shape (14251,), dtype int8
- account_ages array: shape (14251,), dtype float32


### 🔹 D Slicing: `arr[start:stop:step]`
- **What it does:** Extracts a sub-sequence of numerical values.
- **Syntax:** `arr[start:stop:step]`
- **Key Note:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.
- **Dataset Application & Code Demonstration:** Applies D Slicing across the extracted numeric transaction `amounts` array to compute performance metrics.


In [2]:
print('1D Sliced Amounts (first 5 even indices):', amounts[0:10:2])

1D Sliced Amounts (first 5 even indices): [ 607.78   64.08  772.74  217.23 1320.66]


### 🔹 Multi-Axis 2D Slicing
- **What it does:** Slices rows and columns from a multi-feature elements matrix.
- **Syntax:** `function(*args, **kwargs)`
- **Key Note:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.
- **Dataset Application & Code Demonstration:** Applies Multi-Axis 2D Slicing across the extracted numeric transaction `amounts` array to compute performance metrics.


In [3]:
tx_matrix = np.column_stack([amounts[:100], account_ages[:100]])
print('2D Sliced Matrix (Rows 0-3, Cols 0-1):\n', tx_matrix[0:3, 0:2])

2D Sliced Matrix (Rows 0-3, Cols 0-1):
 [[ 607.78    8.  ]
 [1819.11   28.  ]
 [  64.08   91.  ]]


### 🔹 View Checking with `.base`
- **What it does:** Base object if memory is from some other object (view), or None if array owns its memory buffer.
- **Syntax:** `ndarray.base`
- **Key Note:** If `arr.base is not None`, modifying `arr` mutates the underlying original array.
- **Dataset Application & Code Demonstration:** Demonstrates View Checking with `.base` with practical fintech data structures and variables in the following code block.


In [4]:
slice_view = tx_matrix[0:5, 0]
copy_view = tx_matrix[0:5, 0].copy()
print('slice_view shares memory (is view)?:', slice_view.base is not None)
print('copy_view is independent copy?:', copy_view.base is None)

slice_view shares memory (is view)?: True
copy_view is independent copy?: True


### 🔹 Vectorized Boolean Masking
- **What it does:** Filters all fraudulent high-value elements (> $500).
- **Syntax:** `function(*args, **kwargs)`
- **Key Note:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.
- **Dataset Application & Code Demonstration:** Applies Vectorized Boolean Masking across the extracted numeric transaction `amounts` array to compute performance metrics.


In [5]:
fraud_mask = (fraud_flags == 1) & (amounts > 500.0)
high_val_fraud = amounts[fraud_mask]
print(f'High-Value Fraud Amounts Found ({len(high_val_fraud)} txs):', high_val_fraud[:5].round(2))

High-Value Fraud Amounts Found (1572 txs): [1819.11 1998.8  1806.44 1916.91 1911.12]


### 🔹 Fancy Indexing with Integer Arrays
- **What it does:** Extracts elements at specific non-contiguous index locations into a deep copy.
- **Syntax:** `function(*args, **kwargs)`
- **Key Note:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.
- **Dataset Application & Code Demonstration:** Applies Fancy Indexing with Integer Arrays across the extracted numeric transaction `amounts` array to compute performance metrics.


In [6]:
fancy_sample = amounts[[0, 42, 100, 500]]
print('Fancy Indexed Amounts:', fancy_sample)
print('Is Fancy Indexing an isolated copy?:', fancy_sample.base is None)

Fancy Indexed Amounts: [ 607.78 1242.54  518.7  1678.55]
Is Fancy Indexing an isolated copy?: True


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: In-Place Outlier Clamping via Boolean Masking
- **Objective:** Q1: In-Place Outlier Clamping via Boolean Masking
- **Approach:** Clamp all transaction amounts exceeding $2000 down to $2000 in-place without memory allocation.
- **Syntax:** `amounts[amounts > 2000.0] = 2000.0`

In [7]:
clamped_amounts = amounts.copy()
clamped_amounts[clamped_amounts > 2000.0] = 2000.0
print('Max Amount After In-Place Clamping:', clamped_amounts.max())

Max Amount After In-Place Clamping: 1999.98
